# 🧠 Texturizar TU modelo 3D con TU imagen — MV-Adapter (gratis, IA multi-vista)

El proyecto especializado que buscabas: **[MV-Adapter](https://github.com/huanngzh/MV-Adapter)** (ICCV 2025, usado por VAST-AI).
Le das **tu malla 3D existente** (el `.glb` de Hunyuan) + **tu imagen de referencia**, y una IA de difusión genera la textura
**desde 6 vistas a la vez** (consistente alrededor de todo el modelo) y la hornea sobre la malla → sale un `.glb` texturizado.

- No es proyección: la IA "imagina" coherentemente los costados y la espalda a partir de tu imagen.
- Pide ~14 GB de GPU → **entra justo en la T4 gratis** (cerrá otros notebooks para no compartir memoria).
- No requiere cuenta ni token (los modelos que baja son públicos).

## Orden
1. **GPU T4**: `Entorno de ejecución` → `Cambiar tipo de entorno` → **T4 GPU** → Guardar.
2. **Celda 1** (instalar, ~6-10 min: compila nvdiffrast).
3. **Celda 2** (subir tu `.glb`) → **Celda 3** (subir tu imagen).
4. **Celda 4** (texturizar, la 1ª vez baja varios GB de modelos) → **Celda 5** (descargar).

## Celda 1 — Instalar MV-Adapter
Compila `nvdiffrast` (rasterizador de NVIDIA): tarda unos minutos. Ignorá warnings amarillos.

In [ ]:
!nvidia-smi -L

import os
os.chdir('/content')
if not os.path.isdir('/content/MV-Adapter'):
    !git clone https://github.com/huanngzh/MV-Adapter.git
os.chdir('/content/MV-Adapter')

!pip install -q -r requirements.txt 2>&1 | tail -4

import torch
print('\ntorch:', torch.__version__, '| GPU:', torch.cuda.is_available())
try:
    import nvdiffrast, diffusers, open3d, pymeshlab
    print('✅ nvdiffrast + diffusers + open3d + pymeshlab OK. Seguí con la Celda 2.')
except Exception as e:
    print('⚠️ Falta algo:', e, '— copiame este error.')

## Celda 2 — Subir tu modelo 3D (.glb)
El `hunyuan_mesh_2.glb` (la malla gris). Si el botón no anda en el celular, subilo por el panel **Archivos** 📁 y corré la celda igual.

In [ ]:
import os, glob

MESH = None
try:
    from google.colab import files
    up = files.upload()
    if up:
        MESH = os.path.abspath(list(up.keys())[0])
except Exception as e:
    print('El widget no anduvo (', e, ') -> uso el panel Archivos.')

if not MESH or not os.path.exists(MESH):
    cand = glob.glob('/content/*.glb') + glob.glob('/content/*.obj')
    cand = [c for c in cand if '/MV-Adapter/' not in c]
    cand.sort(key=os.path.getmtime)
    MESH = cand[-1] if cand else None

assert MESH and os.path.exists(MESH), 'No encontré el modelo. Subilo por el botón o por el panel Archivos 📁 y volvé a correr esta celda.'
print('Modelo:', MESH, '| ✅ Seguí con la Celda 3.')

## Celda 3 — Subir tu imagen de referencia
La del personaje **de frente** (mejor PNG sin fondo; `--remove_bg` igual lo quita solo).

In [ ]:
import os, glob
from PIL import Image

IMG = None
try:
    from google.colab import files
    up = files.upload()
    if up:
        IMG = os.path.abspath(list(up.keys())[0])
except Exception as e:
    print('El widget no anduvo (', e, ') -> uso el panel Archivos.')

if not IMG or not os.path.exists(IMG):
    cand = []
    for ext in ('png','jpg','jpeg','webp'):
        cand += glob.glob('/content/*.'+ext)
    cand.sort(key=os.path.getmtime)
    IMG = cand[-1] if cand else None

assert IMG and os.path.exists(IMG), 'No encontré la imagen. Subila y volvé a correr esta celda.'
im = Image.open(IMG)
print('Imagen:', IMG, '| modo:', im.mode, '| tamaño:', im.size)
print('✅ Seguí con la Celda 4.')

## Celda 4 — Texturizar (IA multi-vista)
La **primera vez baja varios GB** (SDXL + el adapter) → paciencia. Después tarda unos minutos por modelo.
El resultado queda en `outputs/resultado_shaded.glb`.

In [ ]:
import os
os.chdir('/content/MV-Adapter')
os.makedirs('outputs', exist_ok=True)

!python -m scripts.texture_i2tex --image "{IMG}" --mesh "{MESH}" --save_dir outputs --save_name resultado --remove_bg

OUT = None
for root, dirs, fs in os.walk('outputs'):
    for f in fs:
        if f.endswith('.glb'):
            OUT = os.path.join(root, f)
print('\nResultado:', ('✅ ' + OUT + ' — ' + str(round(os.path.getsize(OUT)/1024/1024, 2)) + ' MB')
      if OUT else '❌ no se generó — copiame el error rojo de arriba')

## Celda 5 — Descargar
Probalo en https://gltf-viewer.donmccurdy.com — tu malla, con la textura de tu imagen alrededor de todo el modelo.

In [ ]:
from google.colab import files
import os
for root, dirs, fs in os.walk('/content/MV-Adapter/outputs'):
    for f in fs:
        if f.endswith('.glb'):
            files.download(os.path.join(root, f))

---
### Si algo falla
- **`CUDA out of memory`** (Celda 4): la T4 va justa. `Entorno de ejecución → Reiniciar sesión`, corré SOLO Celdas 2-3-4 (sin gastar memoria en otra cosa). Si persiste, probá con una malla más liviana (te puedo decimar el .glb) o usá el Space online https://huggingface.co/spaces/VAST-AI/MV-Adapter-Img2Texture (misma IA, GPU de Hugging Face, gratis con cola).
- **Error compilando nvdiffrast** (Celda 1): `!apt-get -qq install -y ninja-build` y repetí la Celda 1.
- **`ModuleNotFoundError`**: copiame el nombre del módulo y te doy el `pip install` exacto.
- **Tarda mucho bajando modelos**: normal la primera vez (SDXL pesa ~7 GB).
- Cualquier error rojo, copiámelo. 🧠